# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [42]:
# ## 1. Method Choice and Why

# For this lane, I will use a Random Forest model for refresh opportunity scoring.

# The baseline created in Week 5 uses manually defined rules based on low CTR and content staleness. However, content refresh decisions depend on multiple signals such as search visibility, engagement, ranking performance, freshness, and traffic trends.

# A Random Forest model is suitable because it can learn relationships between multiple features and identify patterns that may not be captured by a fixed rule.

# The objective is not to prove that refreshing a page will improve performance. Instead, the model provides a better decision-support ranking of pages that may require content review.

In [43]:
import pandas as pd

feature_df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

feature_df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [44]:
feature_df.shape

(30000, 44)

In [45]:
feature_df.columns

Index(['content_id', 'client_id', 'search_volume', 'competition',
       'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count',
       'char_count', 'provider_used', 'model_used', 'impressions_90d',
       'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
       'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
       'days_with_impressions', 'days_with_sessions', 'impressions_last_30d',
       'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
       'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier',
       'age_tier_order', 'days_since_last_update', 'freshness_tier',
       'word_count_tier', 'char_count_tier', 'ctr', 'avg_position',
       'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier',
       'position_tier', 'trend_direction', 'trend_pct'],
      dtype='object')

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [46]:
# ## 2. Split Design

# The dataset does not contain a timestamp that allows a true time-based split. Each row represents aggregated historical performance for one content page over a fixed observation window.

# Therefore, an 80:20 train-test split is used. The split is performed after creating the proxy label and uses a fixed random state for reproducibility.

# The same dataset and proxy target are used for both the baseline and the machine learning model so that the comparison is fair.

In [47]:
feature_df["refresh_needed"] = (
    (feature_df["ctr"] < feature_df["ctr"].median()) &
    (feature_df["days_since_last_update"] > feature_df["days_since_last_update"].median())
).astype(int)

feature_df["refresh_needed"].value_counts()

,count
refresh_needed,
0,23458
1,6542


In [48]:
features = [
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "engaged_sessions_90d",
    "search_volume",
    "ctr",
    "avg_position",
    "engagement_rate",
    "content_age_days",
    "days_since_last_update",
    "trend_pct"
]

X = feature_df[features]
y = feature_df["refresh_needed"]

In [49]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [50]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [51]:
y_pred = model.predict(X_test)

y_prob = model.predict_proba(X_test)[:,1]

In [52]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4692
           1       1.00      1.00      1.00      1308

    accuracy                           1.00      6000
   macro avg       1.00      1.00      1.00      6000
weighted avg       1.00      1.00      1.00      6000



In [53]:
baseline_pred = (
    (X_test["ctr"] < feature_df["ctr"].median()) &
    (X_test["days_since_last_update"] > feature_df["days_since_last_update"].median())
).astype(int)

In [54]:
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd

comparison = pd.DataFrame({
    "Method": ["Week 5 Baseline", "Random Forest"],
    "Precision": [
        precision_score(y_test, baseline_pred),
        precision_score(y_test, y_pred)
    ],
    "Recall": [
        recall_score(y_test, baseline_pred),
        recall_score(y_test, y_pred)
    ],
    "F1 Score": [
        f1_score(y_test, baseline_pred),
        f1_score(y_test, y_pred)
    ]
})

comparison

,Method,Precision,Recall,F1 Score
0,Week 5 Baseline,1.0,1.0,1.0
1,Random Forest,1.0,1.0,1.0


In [55]:
importance = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
})

importance.sort_values("Importance", ascending=False)

,Feature,Importance
10,days_since_last_update,0.417355
6,ctr,0.259377
1,clicks_90d,0.135310
9,content_age_days,0.051773
0,impressions_90d,0.041411
7,avg_position,0.030746
3,sessions_90d,0.015964
11,trend_pct,0.015404
2,pageviews_90d,0.014403
8,engagement_rate,0.007285


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [57]:
importance = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

importance

,Feature,Importance
10,days_since_last_update,0.417355
6,ctr,0.259377
1,clicks_90d,0.135310
9,content_age_days,0.051773
0,impressions_90d,0.041411
7,avg_position,0.030746
3,sessions_90d,0.015964
11,trend_pct,0.015404
2,pageviews_90d,0.014403
8,engagement_rate,0.007285


In [58]:
results = X_test.copy()

results["Actual"] = y_test.to_numpy()
results["Predicted"] = y_pred

false_positive = results[
    (results["Actual"] == 0) &
    (results["Predicted"] == 1)
]

false_negative = results[
    (results["Actual"] == 1) &
    (results["Predicted"] == 0)
]

print("False Positives:", len(false_positive))
print("False Negatives:", len(false_negative))

False Positives: 0
False Negatives: 0


In [56]:
# ## 4. Errors and Interpretation

# The Random Forest model achieved zero false positives and zero false negatives on the test set. This indicates that the model successfully learned the proxy refresh rule used to generate the training labels.

# This result should be interpreted carefully. The target labels are not real records of whether a page required a refresh. Instead, they were created using the Week 5 baseline rule based on CTR and days since the last update.

# Therefore, the model is mainly learning to reproduce the existing rule rather than discovering new patterns from real refresh outcomes.

# The feature importance analysis shows which input features contributed most to the predictions. Features such as CTR, days since last update, engagement rate, and search visibility had the greatest influence on the model's decisions.

# # Although the model performs perfectly on the proxy labels, future work should evaluate it using actual refresh outcomes if those labels become available.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.